# Return State Discovery 01 — Bayesian nonparametric HDP-HSMM on price-return paths

**Exploratory only; no number of regimes prespecified.** This notebook discovers candidate persistent **return/price** behavior in *non-overlapping* 30-minute BTC and ETH return blocks, standardized by forecasts from the executed VD18 model. The forecast at the beginning of a block cannot observe any of that block's returns; state filtering uses the completed block only for the **next** forecast.

Protocol: [`RSD 001`](../../discussions/Return_State_Discovery/001_chatgpt_hdp_hsmm_return_states_preregistration.md).

The finite weak-limit `Kcap=12` is a computational truncation, not a model-selected state count. Transitions exclude self *between episodes*; state persistence is controlled by **explicit shifted-negative-binomial durations**, rather than adding an unidentified sticky diagonal on top of durations. The sampler is not an exact infinite-state HDP posterior. Posterior occupied K **alone does not establish regimes**, particularly because nonlinear price-path features can produce clusters under a regime-free Student-t null.

**Historical-test reuse disclosure:** VD18's original price TRAIN/VALID precede the 150-day historical TEST, but this period has already been inspected across prior modules. The internal HSMM chronology is exploratory, not an untouched prospective confirmation.

## 0. Environment, source test, and paths

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Open notebook from within repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_har_variance as v18
from data_engineering.modeling import hdp_hsmm_return_states as rsd
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Missing Binance 5-minute price files: {HIST}'
OUT=ROOT/'notebooks'/'Return_State_Discovery'/'01';OUT.mkdir(parents=True,exist_ok=True)
ASSETS=('BTC','ETH')
START=time.monotonic()
smoke=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_hdp_hsmm_return_states.py')],
  cwd=str(ROOT),capture_output=True,text=True)
print(smoke.stdout[-2500:]);print(smoke.stderr[-500:]);assert smoke.returncode==0
print('RSD source version:',rsd.VERSION,'| Historical prior result selection: exploratory')

............                                                             [100%]
12 passed in 19.78s


RSD source version: RSD01-0.1.0 | Historical prior result selection: exploratory


## 1. Reconstruct VD18 AM-T-HAR-Blend from pre-test history only

In [2]:
raw,excluded=v14.load_returns(HIST,ASSETS)
panel=v14.panelize(raw)
del raw
feat=v14.make_features(panel)
frame=v14.make_targets(panel,feat,v18.HORIZONS)
frame=v18.add_har_lags(frame)
print('Assets:',frame.underlying.unique(), '| excluded:',excluded)
assert set(frame.underlying.unique())==set(ASSETS)
# This refits native Student-t, Markov factor propagation, global shape/scale,
# HAR forecast and VALID-fitted blend without importing prior fitted results.
blocks,baseline_contract=rsd.build_historical_30m_blocks(frame,particles=256,cal_limit=512)
blocks=blocks.sort_values(['underlying','state_time']).reset_index(drop=True)
assert blocks[['feature_1','feature_2','feature_3','pred_var','baseline_nll']].notna().all().all()
print('AM-T-HAR-Blend audit:',json.dumps(baseline_contract,indent=2))
print(blocks.groupby('underlying').agg(n=('R_30','size'),first=('state_time','min'),last=('state_time','max')))
print('Elapsed:',round(time.monotonic()-START,1),'seconds')

Assets: <ArrowStringArray>
['BTC', 'ETH']
Length: 2, dtype: str | excluded: {}
AM-T-HAR-Blend audit: {
  "har_family": "HAR_4",
  "blend_weight": 0.8395717427385103,
  "base_scale": 1.4756492057728403,
  "base_shape": 1.826083232241286,
  "historical_test_origin_count": 14398
}
               n                     first                      last
underlying                                                          
BTC         7199 2026-04-22 00:00:00+00:00 2026-09-18 23:00:00+00:00
ETH         7199 2026-04-22 00:00:00+00:00 2026-09-18 23:00:00+00:00
Elapsed: 37.2 seconds


## 2. Chronological discovery, validation and diagnostic TEST partitions

These are subdivisions of **VD18's previously inspected historical TEST**, not independent out-of-sample confirmation. The HSMM is fit on approximately its first 90 days and compared sequentially over the last 30 days. The middle 30 days can be reviewed for diagnostics, but do not tune to it then claim the final period pristine.

In [3]:
days=sorted(pd.to_datetime(blocks.state_time,utc=True).dt.floor('D').unique())
n_days=len(days);cut1=days[int(n_days*.6)];cut2=days[int(n_days*.8)]
blocks['period']=np.where(blocks.state_time<cut1,'fit',np.where(blocks.state_time<cut2,'diagnostic','evaluation'))
print('HSMM chronological boundaries:',str(cut1),str(cut2),'| dates:',n_days)
print(blocks.groupby(['underlying','period']).agg(n=('R_30','size'),days=('state_time',lambda v:v.dt.floor('D').nunique())).to_string())
FEATURES=['feature_1','feature_2','feature_3']

def with_contiguous_segments(group,min_len):
    group=group.sort_values('state_time').reset_index(drop=True)
    pieces=rsd.contiguous_segments(group.state_time,min_length=min_len)
    if not pieces:raise ValueError('No adequate contiguous blocks for this asset/period')
    index=np.concatenate([np.arange(a,b) for a,b in pieces]);clean=group.iloc[index].reset_index(drop=True)
    bounds=np.r_[0,np.cumsum([b-a for a,b in pieces])].astype(int)
    return clean,bounds

split_data={}
for asset in ASSETS:
    ga=blocks.loc[blocks.underlying==asset]
    tr,ts=with_contiguous_segments(ga.loc[ga.period=='fit'],min_len=24)
    ev,es=with_contiguous_segments(ga.loc[ga.period=='evaluation'],min_len=1)
    dg,ds=with_contiguous_segments(ga.loc[ga.period=='diagnostic'],min_len=1)
    norm=rsd.fit_normalizer(tr[FEATURES].to_numpy())
    split_data[asset]={'fit':tr,'fit_bounds':ts,'eval':ev,'eval_bounds':es,
                       'diagnostic':dg,'diag_bounds':ds,'normalizer':norm,
                       'X_fit':rsd.apply_normalizer(tr[FEATURES].to_numpy(),norm),
                       'X_eval':rsd.apply_normalizer(ev[FEATURES].to_numpy(),norm)}
    print(asset, 'train blocks:',len(tr),'train segments:',len(ts)-1,
          'evaluation blocks:',len(ev),'evaluation segments:',len(es)-1)


HSMM chronological boundaries: 2026-07-21 00:00:00+00:00 2026-08-20 00:00:00+00:00 | dates: 150
                          n  days
underlying period                
BTC        diagnostic  1440    30
           evaluation  1439    30
           fit         4320    90
ETH        diagnostic  1440    30
           evaluation  1439    30
           fit         4320    90
BTC train blocks: 4320 train segments: 1 evaluation blocks: 1439 evaluation segments: 1
ETH train blocks: 4320 train segments: 1 evaluation blocks: 1439 evaluation segments: 1


## 3. Fit the Bayesian nonparametric HSMM — no fixed K selected

At least two independent chains are fitted per asset. We report posterior occupancy, its sensitivity to initial states/priors, global beta MH acceptance, and duration-cap diagnostics. A high occupied-state count is *not* a positive discovery by itself.

In [4]:
spec_base=rsd.Spec(kcap=12,dmax=96,alpha=3.,gamma=.7,df=6.,
                   n_iter=140,burn=80,thin=4,seed=1801)
results={}; state_rows=[]; chain_rows=[]
for asset in ASSETS:
    data=split_data[asset];x=data['X_fit'];bounds=data['fit_bounds']
    chains=[]
    for ci,seed in enumerate((1801,2909),start=1):
        sp=rsd.Spec(**{**spec_base.__dict__,'seed':seed})
        posterior=rsd.fit_posterior(x,sp,segments=bounds)
        summ=rsd.posterior_summary(posterior)
        chain_rows.append({'asset':asset,'chain':ci,**{k:v for k,v in summ.items() if k!='K_histogram'},
                           'K_histogram':json.dumps(summ['K_histogram'])})
        print(asset,'chain',ci,'K posterior:',summ['K_histogram'],
              'MH:',round(summ['beta_mh_acceptance'],3),'cap warning:',summ['warning'])
        chains.append(posterior)
    # State labels only for a single representative posterior draw, with no
    # claim of label alignment or unique K across chains.
    representative=chains[0]['draws'][len(chains[0]['draws'])//2]
    occupied=np.where(representative['occupancy']>0)[0]
    ordered=sorted(occupied,key=lambda k:representative['mu'][k,0])
    for label,k in enumerate(ordered):
        lens=representative['episode_lengths'][representative['episode_states']==k]
        state_rows.append({'asset':asset,'state_rank_in_draw':label,'occupancy':int(representative['occupancy'][k]),
                           'mean_return_surprise_normalized':float(representative['mu'][k,0]),
                           'mean_path_efficiency_normalized':float(representative['mu'][k,1]),
                           'mean_within_block_dependence_normalized':float(representative['mu'][k,2]),
                           'episode_count':len(lens),'median_episode_blocks':float(np.median(lens)) if len(lens) else np.nan})
    results[asset]={'chains':chains}
chain_table=pd.DataFrame(chain_rows)
state_table=pd.DataFrame(state_rows)
display(chain_table)
display(state_table.round(4))
print('Any Kcap/duration cap warning forbids interpretation of estimated state count or durations.')

BTC chain 1 K posterior: {'4': 5, '5': 6, '6': 3, '7': 1} MH: 0.65 cap warning: NONE
BTC chain 2 K posterior: {'5': 1, '6': 12, '7': 2} MH: 0.586 cap warning: NONE
ETH chain 1 K posterior: {'5': 14, '6': 1} MH: 0.65 cap warning: NONE
ETH chain 2 K posterior: {'5': 15} MH: 0.607 cap warning: NONE


,asset,chain,n_draws,K_occupied_median,K_occupied_05_95,at_Kcap_probability,at_duration_cap_mean,beta_mh_acceptance,warning,K_histogram
0,BTC,1,15,5.0,"[4.0, 6.299999999999999]",0.0,0.0,0.650000,NONE,"{""4"": 5, ""5"": 6, ""6"": 3, ""7"": 1}"
1,BTC,2,15,6.0,"[5.7, 7.0]",0.0,0.0,0.585714,NONE,"{""5"": 1, ""6"": 12, ""7"": 2}"
2,ETH,1,15,5.0,"[5.0, 5.299999999999999]",0.0,0.0,0.650000,NONE,"{""5"": 14, ""6"": 1}"
3,ETH,2,15,5.0,"[5.0, 5.0]",0.0,0.0,0.607143,NONE,"{""5"": 15}"


,asset,state_rank_in_draw,occupancy,mean_return_surprise_normalized,mean_path_efficiency_normalized,mean_within_block_dependence_normalized,episode_count,median_episode_blocks
0,BTC,0,51,-0.9431,-0.6145,-0.7340,20,2.5
1,BTC,1,2237,-0.0235,-0.0292,-0.4924,1030,2.0
2,BTC,2,1776,0.0224,0.0088,0.3540,1008,1.0
3,BTC,3,253,0.2565,0.3723,1.5658,246,1.0
4,BTC,4,3,3.8834,1.9728,0.5222,2,1.5
5,ETH,0,408,-1.7438,-1.1020,0.1047,313,1.0
6,ETH,1,1566,0.0225,-0.0249,0.1157,918,1.0
7,ETH,2,1739,0.0487,0.0511,-0.5587,1079,1.0
8,ETH,3,336,0.4137,0.5693,1.2335,295,1.0
9,ETH,4,271,2.1715,1.3643,0.2672,198,1.0


Any Kcap/duration cap warning forbids interpretation of estimated state count or durations.


## 4. Causal future-return NLL: HSMM versus frozen AM-T-HAR-Blend and non-switching Student-t

For each evaluation block, score its signed net return **before** observing that block's efficiency/reversal features. Only after scoring is the complete observed feature vector used to update state probabilities. The state-free Student-t comparator is fitted on the same recent HSMM training sample, controlling for the advantage of re-estimating return calibration on more recent dates.

HSMM and state-free Student-t return scores are transformed from normalized return to actual return density with the exact Jacobian. Joint feature-vector scores are recorded separately; their units are not comparable to return NLL.

In [5]:
score_rows=[]; evaluated={}
for asset in ASSETS:
    d=split_data[asset];train=d['fit'];ev=d['eval'];xtrain=d['X_fit'];xev=d['X_eval']
    nullfit=rsd.fit_null_t(xtrain,df=spec_base.df)
    # Average *densities* across separately seeded posterior chains, not mean NLL.
    hsmm_log=np.vstack([rsd.predict_return_logscore(xev,chain,segments=d['eval_bounds'])
                        for chain in results[asset]['chains']])
    from scipy.special import logsumexp
    hsmm_log=logsumexp(hsmm_log,axis=0)-np.log(len(results[asset]['chains']))
    t_log=rsd.null_return_logscore(xev,nullfit)
    log_jac=np.log(np.sqrt(ev.pred_var.to_numpy())*d['normalizer']['scale'][0])
    score=ev[['underlying','state_time','baseline_nll','pred_var']].copy()
    score['hdp_hsmm_nll']=-hsmm_log+log_jac
    score['single_t_nll']=-t_log+log_jac
    score['date']=score.state_time.dt.floor('D')
    score['delta_hsmm_vs_har']=score.hdp_hsmm_nll-score.baseline_nll
    score['delta_hsmm_vs_single_t']=score.hdp_hsmm_nll-score.single_t_nll
    evaluated[asset]=score
    print(asset,'return NLL:',score[['baseline_nll','single_t_nll','hdp_hsmm_nll']].mean().round(5).to_dict())
    score_rows.append({'asset':asset,'eval_origins':len(score),
                       'AM_T_HAR_Blend_nll':score.baseline_nll.mean(),
                       'single_t_nll':score.single_t_nll.mean(),
                       'HDP_HSMM_nll':score.hdp_hsmm_nll.mean(),
                       'delta_vs_HAR':score.delta_hsmm_vs_har.mean(),
                       'delta_vs_single_t':score.delta_hsmm_vs_single_t.mean()})
scores=pd.DataFrame(score_rows);display(scores.round(6))

BTC return NLL: {'baseline_nll': -4.60907, 'single_t_nll': -4.61163, 'hdp_hsmm_nll': -4.60752}
ETH return NLL: {'baseline_nll': -4.33751, 'single_t_nll': -4.33111, 'hdp_hsmm_nll': -4.31504}


,asset,eval_origins,AM_T_HAR_Blend_nll,single_t_nll,HDP_HSMM_nll,delta_vs_HAR,delta_vs_single_t
0,BTC,1439,-4.609073,-4.611629,-4.607524,0.001549,0.004104
1,ETH,1439,-4.337505,-4.331114,-4.315036,0.022469,0.016078


## 5. Matched regime-free heavy-tail negative control

Null series have the same **fixed ex-ante variance path**, but independent Student-t(6) innovations within 30-minute blocks. This first null is conditional rather than self-consistent with feedback into HAR. If the real HSMM finds no more structure or predictive gain than these controls, do not claim return regimes. Null simulations intentionally retain nonlinear path-feature geometry.

In [6]:
null_rows=[]
for asset in ASSETS:
    d=split_data[asset];tr=d['fit'];ev=d['eval'];norm=d['normalizer']
    for nullseed in (311,719):
        raw_train=rsd.simulate_price_blocks(len(tr),tr.pred_var.to_numpy(),seed=nullseed,df=6)
        raw_test=rsd.simulate_price_blocks(len(ev),ev.pred_var.to_numpy(),seed=nullseed+1,df=6)
        xtrain=rsd.apply_normalizer(raw_train,norm);xtest=rsd.apply_normalizer(raw_test,norm)
        sp=rsd.Spec(**{**spec_base.__dict__,'seed':nullseed,'n_iter':100,'burn':60,'thin':4})
        post=rsd.fit_posterior(xtrain,sp,segments=d['fit_bounds'])
        single=rsd.fit_null_t(xtrain)
        hpred=rsd.predict_return_logscore(xtest,post,segments=d['eval_bounds'])
        spred=rsd.null_return_logscore(xtest,single)
        summary=rsd.posterior_summary(post)
        null_rows.append({'asset':asset,'seed':nullseed,'null_K_median':summary['K_occupied_median'],
                          'null_K_histogram':json.dumps(summary['K_histogram']),
                          'null_ret_nll_gain_HSMM_vs_single_t':float(np.mean(spred-hpred)),
                          'null_prob_at_Kcap':summary['at_Kcap_probability'],
                          'null_beta_mh_acceptance':summary['beta_mh_acceptance']})
null_table=pd.DataFrame(null_rows);display(null_table.round(5))

,asset,seed,null_K_median,null_K_histogram,null_ret_nll_gain_HSMM_vs_single_t,null_prob_at_Kcap,null_beta_mh_acceptance
0,BTC,311,4.0,"{""4"": 9, ""5"": 1}",0.01112,0.0,0.72
1,BTC,719,4.0,"{""4"": 9, ""5"": 1}",0.03240,0.0,0.70
2,ETH,311,4.0,"{""4"": 10}",0.01444,0.0,0.74
3,ETH,719,4.0,"{""4"": 9, ""5"": 1}",0.03175,0.0,0.66


## 6. Day-block bootstrap, stopping flags, compact outputs

These bootstrap intervals are **historical exploratory**; between-day serial dependence and repeated past model selection remain. The null table is a qualitative diagnostic (two null seeds, not a calibrated frequentist p-value). No raw five-minute returns, full state paths or origin-level posterior files are committed.

In [7]:
rng=np.random.default_rng(1117)
bootstrap=[]
for asset,df in evaluated.items():
    day=df.groupby('date')[['delta_hsmm_vs_har','delta_hsmm_vs_single_t']].mean().to_numpy()
    for j,cmp in enumerate(('HDP_HSMM_minus_AM_T_HAR','HDP_HSMM_minus_recent_single_t')):
        draws=np.mean(day[rng.integers(0,len(day),size=(1000,len(day))),j],axis=1)
        bootstrap.append({'asset':asset,'comparison':cmp,'mean':float(day[:,j].mean()),
                          'day_ci025':float(np.quantile(draws,.025)),
                          'day_ci975':float(np.quantile(draws,.975)),
                          'n_test_days':len(day)})
bootstrap=pd.DataFrame(bootstrap);display(bootstrap.round(6))
checks=[]
for asset in ASSETS:
    q=chain_table[chain_table.asset==asset]
    null=null_table[null_table.asset==asset]
    real_gain=-scores.loc[scores.asset==asset,'delta_vs_single_t'].iloc[0]
    chain_k_spread=float(q.K_occupied_median.max()-q.K_occupied_median.min())
    null_gains=null.null_ret_nll_gain_HSMM_vs_single_t
    checks.append({'asset':asset,'real_gain_vs_recent_t':float(real_gain),
                   'max_null_gain':float(null_gains.max()),
                   'Kcap_warning':bool((q.warning!='NONE').any()),
                   'low_beta_MH_accept':bool((q.beta_mh_acceptance<.05).any()),
                   'chain_K_median_disagreement':chain_k_spread,
                   'duration_cap_rate':float(q.at_duration_cap_mean.max()),
                   'exploratory_candidate':bool(real_gain>max(0.,null_gains.max())
                   and not(q.warning!='NONE').any() and (q.beta_mh_acceptance>=.05).all()
                   and chain_k_spread<=1)})
checks=pd.DataFrame(checks);display(checks)
print('Candidate means *worth investigating*, not a confirmed regime count or options signal.')

,asset,comparison,mean,day_ci025,day_ci975,n_test_days
0,BTC,HDP_HSMM_minus_AM_T_HAR,0.001482,-0.008937,0.011501,30
1,BTC,HDP_HSMM_minus_recent_single_t,0.004080,-0.006058,0.014369,30
2,ETH,HDP_HSMM_minus_AM_T_HAR,0.022476,0.012822,0.032161,30
3,ETH,HDP_HSMM_minus_recent_single_t,0.016091,0.006358,0.026665,30


,asset,real_gain_vs_recent_t,max_null_gain,Kcap_warning,low_beta_MH_accept,chain_K_median_disagreement,duration_cap_rate,exploratory_candidate
0,BTC,-0.004104,0.032405,False,False,1.0,0.0,False
1,ETH,-0.016078,0.031746,False,False,0.0,0.0,False


Candidate means *worth investigating*, not a confirmed regime count or options signal.


## 7. Save reproducibility metadata and aggregates only

In [8]:
for name,table in [('01_scores.csv',scores),('01_posterior_chains.csv',chain_table),
                   ('01_representative_states.csv',state_table),('01_null_controls.csv',null_table),
                   ('01_bootstrap.csv',bootstrap),('01_checks.csv',checks)]:
    table.to_csv(OUT/name,index=False,float_format='%.9g')
manifest={'module':'Return_State_Discovery_01','version':rsd.VERSION,
          'research_status':'HISTORICAL_EXPLORATORY',
          'assets':ASSETS,'original_VD18_test_dates':len(days),
          'HSMM_fit_before':str(cut1),'HSMM_evaluation_from':str(cut2),
          'baseline':baseline_contract,'features':rsd.FEATURES,
          'spec':spec_base.__dict__,'warnings':checks.to_dict('records'),
          'notes':[
            'Posterior occupied K is not confirmed number of economic regimes.',
            'Prior sensitivity and future uncensored-duration inference remain required.',
            'Null controls condition on empirical forecast variance; they are not a fully refitted HAR no-regime null.',
            'State predictions are causal; all constructed path features are known only after block end.',
            'No new options signals, returns trading edge, or independent holdout proof.' ]}
(OUT/'01_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 7 small files under',OUT)
print('Total elapsed seconds:',round(time.monotonic()-START,1))

Saved 7 small files under D:\Projects\deribit_market_data\notebooks\Return_State_Discovery\01
Total elapsed seconds: 264.4


## Interpreting the experiment

1. **Multiple occupied states are not evidence by themselves.** Inspect the matched heavy-tail null K counts, chain dependence and the Kcap warning.
2. **Return likelihood is the primary operational check.** A joint engineered-feature score can improve for reasons unrelated to predicting the next signed return.
3. **State labels are anonymous.** Representative state profiles are from a single posterior draw, not label-aligned across independent chains.
4. **Explicit duration is provisional.** The first/last observed episodes have boundary-censoring bias, and the 48h cap may constrain very persistent states.
5. **Future work, only if supported:** longer posterior chains/prior sensitivity, Kcap extension, full self-consistent no-regime HAR/Student-t simulation, censored HSMM duration inference and genuinely prospective frozen testing.

**No existing model or collector is modified.**